In [1]:
!pip install -q transformers accelerate sentencepiece pandas tqdm


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import json
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

print("Model loaded")

# ---------------------------------------
# LOAD DATA
# ---------------------------------------

df = pd.read_csv("data/train_vulnerable.csv")

# PILOT RUN
# df = df.head(50)

print("Samples:", len(df))

# ---------------------------------------
# GENERATE FIXES
# ---------------------------------------

output_file = "training_pairs.jsonl"

with open(output_file, "w", encoding="utf-8") as outfile:

    for idx, row in tqdm(df.iterrows(), total=len(df)):

        category = str(row["category"])
        cwe = str(row["cwe"])
        code = str(row["source_code"])

        prompt = f"""
You are a senior Java security engineer.

Vulnerability Category: {category}
CWE: CWE-{cwe}

Fix the security vulnerability.

Requirements:
1. Preserve functionality.
2. Use secure coding practices.
3. Return ONLY complete corrected Java code.
4. No markdown.
5. No explanations.

Java Code:

{code}
"""

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=12000
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=2500,
            do_sample=False,
            temperature=0.0
        )

        generated_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        generated_text = generated_text.replace(
            "```java",
            ""
        )
        
        generated_text = generated_text.replace(
            "```",
            ""
        )
        
        generated_text = generated_text.strip()

        record = {
            "instruction": f"Fix CWE-{cwe} {category} vulnerability",
            "input": code,
            "output": generated_text
        }

        outfile.write(
            json.dumps(record, ensure_ascii=False)
            + "\n"
        )

        if (idx + 1) % 5 == 0:
            print(f"Completed {idx + 1}")

print("Done")
print("Saved:", output_file)

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Completed 40



 90%|█████████ | 45/50 [12:00<01:17, 15.42s/it]

Completed 45



100%|██████████| 50/50 [12:58<00:00, 15.56s/it]

Completed 50
Done
Saved: training_pairs.jsonl


In [4]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/
	Output_Generation.ipynb
	training_pairs_50.jsonl

nothing added to commit but untracked files present (use "git add" to track)
